# 05 · Latency and Systems Cost, one machine, one session (work package WP-D)

**Project:** Placement-specific axis reduction of wearable inertial sensors for gym exercise recognition (RecGym), resubmission to *Pervasive and Mobile Computing*.

**Why this notebook exists.** The latency numbers that already exist (`TableS6_efficiency_latency.csv`) come from two different sessions: the classical models in notebook 01 (CPU model not recorded) and the CNN in notebook 02 (Intel Xeon @ 2.00 GHz, 2 cores). They are not comparable with each other, so no systems claim can rest on them. This notebook re-measures **every model on every configuration on the same machine in the same session**, adds model size and complexity counts, an analytic data-volume model, and a Pareto view. It trains nothing that is reported as an accuracy result.

> Nothing is reported until *you* run it. Every number comes from the real run on your Kaggle machine. **Run with `SMOKE_TEST=1` first** (default, a few minutes), then set `SMOKE_TEST=0`.

---

## Frozen design (written before any timing was taken)

| Decision | Value |
|---|---|
| Unit | one (position, configuration, model) |
| Configurations | **30**: the 15 primary within-sensor configurations, the 6 drop-one configurations (A1) and the 9 mixed pairs (A2), i.e. exactly the configurations for which accuracy results exist. Selectable with `CONFIG_GROUPS` (`P,A1,A2`). Primary configurations are asserted equal to `protocol/axis_configs.json`; A1/A2 are asserted against the `channels` column of `extended_full/results.csv` when it is found. |
| Positions | leg (calf), pocket, wrist |
| Models | RF, SVM, XGB, CNN with the **frozen hyper-parameters of notebooks 01 and 02**, seed 42, no tuning, no early stopping. The helper functions are copied unchanged. |
| Trained model per unit | **one**, on the training subjects of fold `LAT_FOLD` (default **0**, held-out subject 1), chosen a priori. Structure-dependent costs (tree depth, support-vector count) can differ across folds; Section 9 compares model size with the 10-fold median of the accuracy grid as a representativeness check. |
| Timed windows | `N_TIME_WINDOWS` = **300** windows spread evenly over the held-out subject's windows, timed in `N_PASSES` = **3** consecutive passes after `N_WARMUP` = 30 untimed calls, i.e. 900 timings per unit (at least 200 required in a full run). |
| Timed pipeline | **Classical:** slice raw window, 15 features per channel, (SVM: `StandardScaler`), `predict`. **CNN:** slice raw window, z-score with the training-fold statistics, forward pass, `argmax`. Preparation and inference are timed separately **and** end to end for the same call. |
| Batch / threads | batch size 1; **1 thread** (`threadpoolctl` limit 1, `torch.set_num_threads(1)`, `n_jobs=1`), CPU only |
| Clock | `time.perf_counter_ns` around every call; garbage collection disabled while timing |
| Statistics | median, IQR, 5th / 95th / 99th percentile, mean of the per-call total; range of the three per-pass medians as a stability indicator |
| Machine guard | `machine_id` = hash of CPU model and core counts; the notebook **refuses to mix machines** in one `results.csv` unless `ALLOW_MIXED_MACHINES=1`. A calibration micro-benchmark is recorded for every unit so drift inside a session is visible. |
| Training device | XGBoost and the CNN may train on a GPU to save time; **all timing is on CPU**. The device is recorded in every row. |
| Model cost | serialized size (`joblib` for RF/SVM/XGB, `torch` state dict for the CNN), exact parameters and MACs for the CNN, node counts for RF and XGB, support-vector counts for SVM |
| Data volume (D2) | analytic: channels × 20 Hz samples per second, at 16-bit and 32-bit per sample, decimal units, raw sample values only |
| Duty cycle | total latency divided by the 2 s hop between consecutive windows: the share of **one core of this CPU** the recogniser would occupy. It is a compute-load figure, **not energy**. |

## Frozen analysis plan

1. **Descriptive only. No hypothesis tests.** Per (position, model, configuration): median and IQR latency, preparation versus inference share, size, complexity counts, change versus the 6-axis configuration of the same position and model.
2. **Pooled by channel count** (1 to 6): median, quartiles and min to max over all units of that size, so the summary does not rest on a best-of selection. Pure and mixed configurations are both included.
3. **Data volume** per configuration and per channel count, with the reduction relative to 6 axes.
4. **Pareto view (D4):** macro-F1 (10-fold mean from the immutable accuracy grid, seed 42) against data rate and against latency, per position and model, with the non-dominated set flagged. It needs `classical_full`, `cnn_full` and `extended_full`; if they are not found it is skipped and the rest still runs.
5. **Quality checks:** stability of the three passes, calibration drift, model size versus the 10-fold grid median, and the fold macro-F1 of the model timed here versus the same fold in the accuracy grid.
6. **D3 (parametric energy sensitivity) is deliberately not part of this notebook.** It would need manufacturer datasheet figures that must be looked up and cited, and nothing here invents them. No energy statement is made anywhere in the outputs.

## What the numbers are, and are not

- They are single-window, batch-1, single-thread CPU latencies of **these software stacks** (scikit-learn, XGBoost, PyTorch) on **one shared, virtualised cloud CPU**. Library call overhead is part of the number (for the 200-tree forest it dominates).
- They are **not edge-device latencies** and **not energy**. They can show how cost changes with channel count on one machine; they cannot show what a given watch, band or phone would do.
- One trained model per unit (one fold): structural variation across folds is not captured here.

## Run control (environment variables, or `RUN_OVERRIDES` in the next cell)

| Variable | Default | Meaning |
|---|---|---|
| `SMOKE_TEST` | `1` | `1`: 4 configurations, leg only, 200 windows x 2 passes, reduced model sizes, into `latency_smoke/`. `0`: full grid into `latency_full/`. |
| `MODELS` | `XGB,CNN,RF,SVM` | Subset and order. |
| `POSITIONS` | all | Chunk by placement, e.g. `wrist`. |
| `CONFIG_GROUPS` | `P,A1,A2` | `P` primary 15, `A1` drop-one 6, `A2` mixed pairs 9. |
| `CONFIG_NAMES` | all selected | Comma-separated configuration names. |
| `LAT_FOLD` | `0` | Fold whose training subjects train the timed models. |
| `N_TIME_WINDOWS`, `N_PASSES`, `N_WARMUP` | `300`, `3`, `30` | Timing sample. |
| `CNN_TRAIN` | `auto` | `auto`: train the CNN if a GPU is visible, otherwise time a fixed-seed untrained network of the same architecture (inference cost does not depend on the weights; recorded in `cnn_weights`). `1`: always train. `0`: never train. |
| `XGB_TRAIN_DEVICE` | `auto` | `auto`: GPU if visible. `cpu` or `cuda` to force. Timing is always on CPU. |
| `TIME_BUDGET_MIN` | none | Stop cleanly after this many minutes, write outputs and zip, resume later (same machine!). |
| `ALLOW_MIXED_MACHINES` | unset | `1` to allow resuming on a different machine. Not recommended: the point of this notebook is consistency. |
| `ANALYZE_PARTIAL` | unset | `1` to build tables and figures from an incomplete run. |
| `REQUIRE_COMPLETE` | unset | `1` to fail at the end if any unit of the selection is missing. |
| `RECGYM_CSV`, `PROTOCOL_DIR`, `PRIMARY_DIR`, `SEARCH_ROOTS`, `RESULTS_DIR` | auto | Paths. On Kaggle everything under `/kaggle/input` is searched automatically. |

## Outputs (in `results/raw/latency_full/`, zipped for download)

`results.csv` (one row per unit), `run_config.json`, `environment.json`, `cpu_info.txt`, `run_log.txt`, `tables/` (`TableL1` to `TableL6`), `figures/` (`FL1` to `FL5`, PNG and PDF), `checks/` (`latency_note.txt`, `latency_qc.json`, `manifest.json`).

## Kaggle set-up

1. Make one Kaggle dataset (or several) that contains `RecGym.csv`, the repository's `protocol/` folder, and the three accuracy-grid folders `classical_full/`, `cnn_full/` and `extended_full/` (each with its `results.csv`). Attach it to the notebook. Everything is found automatically; nothing has to be typed.
2. **Accelerator:** a T4 GPU is optional. It only speeds up XGBoost and CNN training (about 30 minutes saved) and makes the CNN a trained model. Without it the notebook works and uses the untrained-network fallback for the CNN timing. Timing itself always runs on the CPU.
3. Run top to bottom with `SMOKE_TEST=1`, read the timing log, then restart and run with `RUN_OVERRIDES = {"SMOKE_TEST": "0"}`. Plan roughly **2 to 3 hours** for the full grid (360 units; RF training dominates). Keep it in **one session** so that every number comes from the same machine.

In [ ]:
import os, sys, re, json, time, hashlib, platform, random, shutil, zipfile, warnings, itertools, io, copy, gc, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
try:
    display
except NameError:
    display = print

T0 = time.time()

# ---- optional in-notebook overrides (Kaggle: exporting shell variables is awkward). Examples:
#   RUN_OVERRIDES = {"SMOKE_TEST": "0"}                          # full run
#   RUN_OVERRIDES = {"SMOKE_TEST": "0", "POSITIONS": "wrist"}    # one chunk (same machine only!)
RUN_OVERRIDES = {}
os.environ.update({str(k): str(v) for k, v in RUN_OVERRIDES.items()})

def env_list(name, default=None):
    v = os.environ.get(name)
    if v is None or not v.strip():
        return default
    return [x.strip() for x in v.split(",") if x.strip()]

# ============================ USER CONFIG ====================================
SMOKE_TEST = os.environ.get("SMOKE_TEST", "1") == "1"
SEED = 42
FS = 20                       # nominal sampling rate (Hz)
WINDOW_SECONDS = 4
OVERLAP = 0.5                 # fixed a priori; identical to notebooks 01 and 02
EXPECTED_SHA256 = "2031c077a57e93d6b5c39da198470cb6c9b1a3c0652ab57cfc5d6a39bfa840e9"
ENFORCE_HASH = os.environ.get("RECGYM_SKIP_HASH") is None

LAT_FOLD = int(os.environ.get("LAT_FOLD", "0"))
N_TIME_WINDOWS = int(os.environ.get("N_TIME_WINDOWS", "200" if SMOKE_TEST else "300"))
N_PASSES = int(os.environ.get("N_PASSES", "2" if SMOKE_TEST else "3"))
N_WARMUP = int(os.environ.get("N_WARMUP", "30"))

MODELS = [m.upper() for m in env_list("MODELS", ["XGB", "CNN", "RF", "SVM"])]
POSITIONS = env_list("POSITIONS", None)
CONFIG_GROUPS = [g.upper() for g in env_list("CONFIG_GROUPS", ["P", "A1", "A2"])]
CONFIG_NAMES = env_list("CONFIG_NAMES", None)
_tb = os.environ.get("TIME_BUDGET_MIN", "").strip()
TIME_BUDGET_S = float(_tb) * 60.0 if _tb else None
ALLOW_MIXED_MACHINES = os.environ.get("ALLOW_MIXED_MACHINES") == "1"
FORCE_NEW_RUN = os.environ.get("FORCE_NEW_RUN") == "1"
ANALYZE_PARTIAL = os.environ.get("ANALYZE_PARTIAL") == "1"
REQUIRE_COMPLETE = os.environ.get("REQUIRE_COMPLETE") == "1"
CNN_TRAIN = os.environ.get("CNN_TRAIN", "auto").lower()
XGB_TRAIN_DEVICE = os.environ.get("XGB_TRAIN_DEVICE", "auto").lower()

# hyper-parameters exactly as notebooks 01 / 02 (frozen)
RF_PARAMS = dict(n_estimators=200, max_depth=None, min_samples_leaf=1, max_features="sqrt",
                 class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
SVM_PARAMS = dict(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", cache_size=1000, random_state=SEED)
SVM_MAX_TRAIN_WINDOWS = 15000
XGB_PARAMS = dict(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                  tree_method="hist", n_jobs=-1, random_state=SEED)
CNN_PARAMS = dict(widths=(32, 64, 128), kernels=(5, 5, 3), dropout=0.3)
TRAIN_PARAMS = dict(epochs=25, batch_size=256, max_lr=3e-3, weight_decay=1e-3)
# ==============================================================================

SMOKE_CONFIGS = ["6-axis", "gyro-GxGz", "single-Gz", "mix-AxGz"]    # one per size class plus one mixed pair
if SMOKE_TEST:
    POSITIONS = POSITIONS or ["leg"]
    CONFIG_NAMES = CONFIG_NAMES or list(SMOKE_CONFIGS)
    RF_PARAMS = {**RF_PARAMS, "n_estimators": 30}
    XGB_PARAMS = {**XGB_PARAMS, "n_estimators": 30}
    SVM_MAX_TRAIN_WINDOWS = 4000
    TRAIN_PARAMS = {**TRAIN_PARAMS, "epochs": 3}

assert all(m in ("RF", "SVM", "XGB", "CNN") for m in MODELS), f"unknown MODELS: {MODELS}"
assert len(set(MODELS)) == len(MODELS), "duplicates in MODELS"
assert all(g in ("P", "A1", "A2") for g in CONFIG_GROUPS), f"unknown CONFIG_GROUPS: {CONFIG_GROUPS}"
assert CNN_TRAIN in ("auto", "0", "1"), "CNN_TRAIN must be auto, 0 or 1"
assert XGB_TRAIN_DEVICE in ("auto", "cpu", "cuda"), "XGB_TRAIN_DEVICE must be auto, cpu or cuda"
assert N_PASSES >= 1 and N_WARMUP >= 0 and N_TIME_WINDOWS >= 1
if not SMOKE_TEST:
    assert N_TIME_WINDOWS * N_PASSES >= 200, "a full run needs at least 200 timings per unit"

IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
BASE = Path("/kaggle/working") if IN_KAGGLE else (Path("/content") if IN_COLAB else Path(".").resolve())
random.seed(SEED); np.random.seed(SEED)

RESULTS_ROOT = Path(os.environ.get("RESULTS_DIR", str(BASE / "results" / "raw")))
TAG = "smoke" if SMOKE_TEST else "full"
RUN_DIR = RESULTS_ROOT / f"latency_{TAG}"
TABLE_DIR, FIG_DIR, CHECK_DIR = RUN_DIR / "tables", RUN_DIR / "figures", RUN_DIR / "checks"
for d in [RUN_DIR, TABLE_DIR, FIG_DIR, CHECK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

LOG = []
def log(*a):
    s = " ".join(str(x) for x in a); print(s); LOG.append(s)

from threadpoolctl import threadpool_limits
try:
    import torch
    import torch.nn as nn
    HAVE_TORCH = True
except Exception as _e:
    HAVE_TORCH = False
    log("PyTorch not importable:", _e)
assert HAVE_TORCH or "CNN" not in MODELS, "CNN requested but PyTorch is not available"
DEVICE = torch.device("cuda" if (HAVE_TORCH and torch.cuda.is_available()) else "cpu") if HAVE_TORCH else None
HAS_GPU = shutil.which("nvidia-smi") is not None
XGB_DEVICE_TRAIN = "cuda" if (XGB_TRAIN_DEVICE == "cuda" or (XGB_TRAIN_DEVICE == "auto" and HAS_GPU)) else "cpu"
if XGB_TRAIN_DEVICE == "cuda":
    assert HAS_GPU, "XGB_TRAIN_DEVICE=cuda but no GPU is visible"
if HAVE_TORCH:
    torch.backends.cudnn.benchmark = True      # training speed only; timing is on CPU
CNN_DO_TRAIN = ("CNN" in MODELS) and (CNN_TRAIN == "1" or (CNN_TRAIN == "auto" and DEVICE is not None and DEVICE.type == "cuda"))
CNN_WEIGHTS = "trained" if CNN_DO_TRAIN else "random_init"

log("SMOKE_TEST =", SMOKE_TEST, "| run dir:", RUN_DIR)
log("models:", MODELS, "| fold:", LAT_FOLD, "| timing:", N_TIME_WINDOWS, "windows x", N_PASSES, "passes, warm-up", N_WARMUP)
log("XGB trains on:", XGB_DEVICE_TRAIN, "| CNN weights for timing:", CNN_WEIGHTS, "| torch device:", DEVICE)
if SMOKE_TEST:
    log(">>> SMOKE TEST: configs", SMOKE_CONFIGS, ", positions", POSITIONS, ", reduced model sizes. Set SMOKE_TEST=0 for the real run. <<<")

In [ ]:
# ---- locate inputs automatically (Kaggle attaches datasets under /kaggle/input)
SEARCH_ROOTS = [Path(p) for p in (env_list("SEARCH_ROOTS", None) or ["/kaggle/input", str(BASE), str(Path.cwd())]) if Path(p).exists()]
_SKIP = {"__MACOSX", ".git", "__pycache__", "node_modules"}

def walk_find(match, max_depth=8):
    hits, seen = [], set()
    for root in SEARCH_ROOTS:
        base_depth = len(root.parts)
        for dp, dn, fn in os.walk(root):
            dn[:] = [d for d in dn if d not in _SKIP]
            if len(Path(dp).parts) - base_depth >= max_depth:
                dn[:] = []
            r = match(Path(dp), fn)
            if r is not None and str(r) not in seen:
                seen.add(str(r)); hits.append(r)
    return sorted(hits, key=lambda x: (len(x.parts), str(x)))

def first_or_none(hits, what):
    if len(hits) > 1:
        log(f"NOTE: {len(hits)} candidates for {what}; using the first:", hits[0])
    return hits[0] if hits else None

DATA_PATH = os.environ.get("RECGYM_CSV") or first_or_none(walk_find(lambda d, fn: d / "RecGym.csv" if "RecGym.csv" in fn else None), "RecGym.csv")
PROTOCOL_DIR = os.environ.get("PROTOCOL_DIR") or first_or_none(
    walk_find(lambda d, fn: d if ("axis_configs.json" in fn and "windows_index.parquet" in fn) else None), "protocol/")
assert DATA_PATH and os.path.exists(str(DATA_PATH)), ("RecGym.csv not found. Attach it as a Kaggle dataset or set RECGYM_CSV. "
                                                      f"Searched: {[str(r) for r in SEARCH_ROOTS]}")
assert PROTOCOL_DIR and Path(PROTOCOL_DIR).exists(), ("protocol/ folder (axis_configs.json + windows_index.parquet) not found. "
                                                      "Attach the repository as a dataset or set PROTOCOL_DIR.")
DATA_PATH, PROTOCOL_DIR = str(DATA_PATH), Path(PROTOCOL_DIR)
PRIMARY_DIR = os.environ.get("PRIMARY_DIR", "").strip()

def usable_grid(d):
    # a grid folder is usable for the Pareto view if it holds seed-42 results (or has no seed column at all)
    try:
        hdr = pd.read_csv(d / "results.csv", nrows=0).columns
        if "seed" not in hdr: return True
        return SEED in set(pd.read_csv(d / "results.csv", usecols=["seed"])["seed"].unique())
    except Exception:
        return False

def find_run(name):
    if PRIMARY_DIR:
        for b in [Path(PRIMARY_DIR), Path(PRIMARY_DIR) / "results" / "raw"]:
            if (b / name / "results.csv").exists() and usable_grid(b / name):
                return b / name
    hits = [h for h in walk_find(lambda d, fn: d if (d.name == name and "results.csv" in fn) else None) if usable_grid(h)]
    return first_or_none(hits, name)

GRID_DIRS = {"classical_full": find_run("classical_full"), "cnn_full": find_run("cnn_full"), "extended_full": find_run("extended_full")}
log("RecGym.csv:", DATA_PATH); log("protocol:", PROTOCOL_DIR)
for k, v in GRID_DIRS.items():
    log(f"accuracy grid {k}:", v if v is not None else "NOT FOUND (Pareto view will use what is available)")

In [ ]:
# ---- load the audited data file (same checks as notebooks 01 / 02)
def sha256_of(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

DATA_SHA256 = sha256_of(DATA_PATH)
log("SHA-256:", DATA_SHA256)
if ENFORCE_HASH:
    assert DATA_SHA256 == EXPECTED_SHA256, "The file differs from the audited RecGym.csv (set RECGYM_SKIP_HASH=1 only on purpose)."
    log("Hash matches the audited file.")

raw = pd.read_csv(DATA_PATH)
raw.columns = [str(c).strip() for c in raw.columns]
log("Loaded:", raw.shape, list(raw.columns))

def norm(c): return re.sub(r"[^a-z0-9]", "", str(c).lower())
ncol = {norm(c): c for c in raw.columns}
def pick(cands):
    for k in cands:
        if k in ncol: return ncol[k]
    return None
subj_col = pick(["subject", "object", "subjectid", "volunteer", "participant", "user"])
label_col = pick(["workout", "label", "activity", "exercise"])
pos_col = pick(["position", "placement", "location"])
sess_col = pick(["session", "sessionid", "day"])
IMU = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]
chan_cols = {k: ncol.get(k.lower()[0] + "_" + k.lower()[1]) or ncol.get(k.lower()) for k in IMU}
assert subj_col and label_col and pos_col, f"Cannot detect roles in {list(raw.columns)}"
assert all(chan_cols.values()), f"Cannot detect IMU channels: {chan_cols}"

subject_str = raw[subj_col].astype(str).str.strip()
subjects = sorted(subject_str.unique(), key=lambda s: (len(s), s))
labels = sorted(raw[label_col].astype(str).str.strip().unique())
positions_all = sorted(raw[pos_col].astype(str).str.strip().unique())
K = len(labels)
cat = lambda s, cats: pd.Categorical(s.astype(str).str.strip(), categories=cats).codes.astype(np.int32)
subj_c = cat(raw[subj_col], subjects); pos_c = cat(raw[pos_col], positions_all); lab_c = cat(raw[label_col], labels)
sess_vals = sorted(raw[sess_col].astype(str).str.strip().unique()) if sess_col else ["NA"]
sess_c = cat(raw[sess_col], sess_vals) if sess_col else np.zeros(len(raw), dtype=np.int32)
assert (subj_c >= 0).all() and (pos_c >= 0).all() and (lab_c >= 0).all()
sig = raw[[chan_cols[k] for k in IMU]].to_numpy(dtype=np.float32)
assert np.isfinite(sig).all(), "non-finite IMU values"
n_rows = len(raw); del raw
log("Subjects:", subjects, "| positions:", positions_all, "| labels:", K)

# ---- windows (identical code path to notebook 01) and equality with the saved protocol index
W = int(round(WINDOW_SECONDS * FS)); S = max(1, int(round(W * (1 - OVERLAP))))
HOP_S = S / FS; HOP_MS = 1000.0 * HOP_S
chg = np.ones(n_rows, dtype=bool)
chg[1:] = ((subj_c[1:] != subj_c[:-1]) | (pos_c[1:] != pos_c[:-1]) | (sess_c[1:] != sess_c[:-1]) | (lab_c[1:] != lab_c[:-1]))
starts = np.flatnonzero(chg); lens = np.diff(np.r_[starts, n_rows])
keep = lens >= W
rs, rl = starts[keep], lens[keep]
nwin = (rl - W) // S + 1
tot = int(nwin.sum())
run_of_win = np.repeat(np.arange(len(rs)), nwin)
offs = np.arange(tot) - np.repeat(np.cumsum(nwin) - nwin, nwin)
w_start = (rs[run_of_win] + offs * S).astype(np.int64)
w_end = w_start + W - 1
for nm, arr in [("subject", subj_c), ("position", pos_c), ("session", sess_c), ("label", lab_c)]:
    assert (arr[w_start] == arr[w_end]).all(), f"window crosses a {nm} boundary"
widx = pd.DataFrame({"start_row": w_start, "subject": [subjects[i] for i in subj_c[w_start]],
                     "position": [positions_all[i] for i in pos_c[w_start]], "session": [sess_vals[i] for i in sess_c[w_start]],
                     "label": [labels[i] for i in lab_c[w_start]], "label_id": lab_c[w_start].astype(np.int16), "run_id": rs[run_of_win]})
WINDOWS_SHA256 = hashlib.sha256(w_start.tobytes()).hexdigest()
log(f"Windows: {len(widx)} | window = {W} samples, hop = {S} samples = {HOP_MS:.0f} ms")

ref = pd.read_parquet(PROTOCOL_DIR / "windows_index.parquet")
assert len(ref) == len(widx), f"window count differs from protocol: {len(ref)} vs {len(widx)}"
for c in ["start_row", "label_id", "run_id"]:
    assert (ref[c].to_numpy() == widx[c].to_numpy()).all(), f"window {c} differs from protocol/windows_index.parquet"
for c in ["subject", "position", "session", "label"]:
    assert (ref[c].astype(str).to_numpy() == widx[c].astype(str).to_numpy()).all(), f"window {c} differs from protocol/windows_index.parquet"
meta = json.load(open(PROTOCOL_DIR / "windows_meta.json"))
assert meta["window_samples"] == W and meta["step"] == S and meta["fs"] == FS and meta["labels"] == labels and meta["subjects"] == subjects \
    and meta["data_sha256"] == DATA_SHA256, "protocol/windows_meta.json does not match this run"
del ref
log("VERIFIED: windows equal protocol/windows_index.parquet.")

FOLD_DEFS = json.load(open(PROTOCOL_DIR / "loso_folds.json"))
assert [f["test_subject"] for f in FOLD_DEFS] == subjects and [f["fold"] for f in FOLD_DEFS] == list(range(len(subjects)))
assert 0 <= LAT_FOLD < len(FOLD_DEFS), "LAT_FOLD out of range"
FD = FOLD_DEFS[LAT_FOLD]
log(f"Timed models are trained on fold {LAT_FOLD}: held-out subject {FD['test_subject']}, {len(FD['train_subjects'])} training subjects.")

def gather(rows):                       # raw windows for given start rows -> (n, W, 6)
    return sig[rows[:, None] + np.arange(W)[None, :]]
POS_LIST = POSITIONS or positions_all
assert all(p in positions_all for p in POS_LIST), f"POSITIONS must be among {positions_all}"

## 3 · Features and axis configurations

The feature function is copied unchanged from notebook 01 (15 per-channel features, nothing across channels). The unit test is extended to **every selected configuration**, including the mixed ones: the features of a channel subset must equal the matching columns of the full 6-channel matrix.

In [ ]:
FEATS = ["mean", "std", "min", "max", "median", "iqr", "rms", "skew", "kurt", "mad", "zcr",
         "dom_freq", "spec_energy", "spec_entropy", "spec_centroid"]

def window_features(X, ch_names, fs=FS):
    # X: (N, T, C) float -> (N, C*len(FEATS)) float32, names 'Ax__mean', ...   (unchanged from notebook 01)
    N, T, C = X.shape
    freqs = np.fft.rfftfreq(T, d=1.0 / fs)[1:]
    out, names = [], []
    for c, ch in enumerate(ch_names):
        x = X[:, :, c].astype(np.float64)
        mu = x.mean(1); sd = x.std(1); xc = x - mu[:, None]
        q25, med, q75 = np.percentile(x, [25, 50, 75], axis=1)
        z = xc / np.where(sd > 1e-12, sd, 1.0)[:, None]
        P = np.abs(np.fft.rfft(xc, axis=1))[:, 1:] ** 2
        tot_p = P.sum(1); p = P / np.where(tot_p > 1e-12, tot_p, 1.0)[:, None]
        cols = [mu, sd, x.min(1), x.max(1), med, q75 - q25, np.sqrt((x ** 2).mean(1)),
                (z ** 3).mean(1), (z ** 4).mean(1) - 3.0, np.abs(np.diff(x, axis=1)).mean(1),
                (np.diff(np.signbit(xc).astype(np.int8), axis=1) != 0).mean(1),
                freqs[P.argmax(1)], tot_p / T, -(p * np.log(p + 1e-12)).sum(1), (p * freqs[None, :]).sum(1)]
        out += cols; names += [f"{ch}__{f}" for f in FEATS]
    return np.stack(out, axis=1).astype(np.float32), names

# ---- configurations: primary 15 (asserted against the protocol), A1 drop-one 6, A2 mixed pairs 9
acc, gyr = IMU[:3], IMU[3:]
PRIMARY_CONFIGS = {"6-axis": list(IMU), "acc-3": list(acc), "gyro-3": list(gyr)}
for a, b in itertools.combinations(acc, 2): PRIMARY_CONFIGS[f"acc-{a}{b}"] = [a, b]
for a, b in itertools.combinations(gyr, 2): PRIMARY_CONFIGS[f"gyro-{a}{b}"] = [a, b]
for k in IMU: PRIMARY_CONFIGS[f"single-{k}"] = [k]
assert len(PRIMARY_CONFIGS) == 15
proto_axis = json.load(open(PROTOCOL_DIR / "axis_configs.json"))
assert {k: list(v) for k, v in proto_axis["configs"].items()} == PRIMARY_CONFIGS, "primary configs differ from protocol/axis_configs.json"
A1_CONFIGS = {f"drop-{k}": [c for c in IMU if c != k] for k in IMU}
A2_CONFIGS = {f"mix-{a}{g}": [a, g] for a in acc for g in gyr}
assert len(A1_CONFIGS) == 6 and len(A2_CONFIGS) == 9
GROUP_CONFIGS = {"P": PRIMARY_CONFIGS, "A1": A1_CONFIGS, "A2": A2_CONFIGS}
GROUP_OF, ALL_CONFIGS = {}, {}
for g in CONFIG_GROUPS:
    for k, v in GROUP_CONFIGS[g].items():
        assert k not in ALL_CONFIGS, f"duplicate configuration name {k}"
        ALL_CONFIGS[k] = v; GROUP_OF[k] = g
for k, v in ALL_CONFIGS.items():
    assert v == [c for c in IMU if c in v], f"{k}: channels not in canonical order"
CONFIGS = {k: v for k, v in ALL_CONFIGS.items() if (CONFIG_NAMES is None or k in CONFIG_NAMES)}
_unknown = [c for c in (CONFIG_NAMES or []) if c not in ALL_CONFIGS]
assert not _unknown, f"CONFIG_NAMES not in the selected groups {CONFIG_GROUPS}: {_unknown}"
assert CONFIGS, "no configuration selected"

# ---- cross-check channel lists against the stored accuracy grids (when they are present)
for gname, gdir in GRID_DIRS.items():
    if gdir is None: continue
    _c = pd.read_csv(Path(gdir) / "results.csv", usecols=["config", "channels"]).drop_duplicates()
    _bad = [(r.config, r.channels) for r in _c.itertuples() if r.config in ALL_CONFIGS and "".join(ALL_CONFIGS[r.config]) != r.channels]
    assert not _bad, f"channel lists differ from {gname}/results.csv: {_bad[:5]}"
    log(f"Channel lists agree with {gname}/results.csv for", int(_c.config.isin(list(ALL_CONFIGS)).sum()), "configurations.")
log("Configs this run (", len(CONFIGS), "):", list(CONFIGS))

# ---- per-position arrays and features (only the positions of this run)
t0 = time.time()
POS_DATA = {}
for p in POS_LIST:
    sel = np.flatnonzero(widx["position"].to_numpy() == p)
    Xp = gather(w_start[sel])
    F, fnames = window_features(Xp, IMU)
    subj_p = widx["subject"].to_numpy()[sel]
    D = dict(sel=sel, F=F, names=fnames, y=widx["label_id"].to_numpy()[sel].astype(int), subj=subj_p, starts=w_start[sel])
    if "CNN" in MODELS:
        D["X"] = np.ascontiguousarray(Xp.transpose(0, 2, 1))          # (n, 6, W) for Conv1d
    POS_DATA[p] = D
    log(f"{p}: windows={len(sel)}, feature matrix={F.shape}")
    del Xp
log("Features computed in", round(time.time() - t0, 1), "s")

# ---- unit test: every selected configuration's features == matching columns of the full matrix
_rng = np.random.default_rng(SEED); _rows = w_start[_rng.choice(len(w_start), size=min(300, len(w_start)), replace=False)]
_Xw = gather(_rows)
_full, _names = window_features(_Xw, IMU)
for _cn, _sub in CONFIGS.items():
    _f, _n = window_features(_Xw[:, :, [IMU.index(c) for c in _sub]], _sub)
    _cols = [_names.index(n) for n in _n]
    assert len(_cols) == len(FEATS) * len(_sub)
    assert np.allclose(_f, _full[:, _cols], atol=1e-4, rtol=1e-4), f"feature leakage across channels for {_cn}"
assert np.isfinite(_full).all()
log("Feature unit tests passed for all", len(CONFIGS), "selected configurations (pure and mixed).")
del _Xw, _full

## 4 · Models

RF, SVM and XGBoost: `fit_model`, `predict` and `complexity` are copied from notebook 01 (the per-sample balanced weights are named `balanced_sample_weights` here to keep them apart from the CNN's per-class weights). The 1D CNN, `train_cnn`, `count_macs` and the size helpers are copied from notebook 02. Only the timing harness is new; it replaces `measure_latency`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import joblib
if "XGB" in MODELS:
    try:
        import xgboost as xgb
    except ImportError:
        os.system(f"{sys.executable} -m pip install -q xgboost"); import xgboost as xgb

def balanced_sample_weights(y, K):
    cnt = np.bincount(y, minlength=K).astype(float)
    w = np.where(cnt > 0, len(y) / (K * np.maximum(cnt, 1)), 0.0)
    return w[y]

def fit_model(name, Xtr, ytr, fold):
    # returns (model, scaler_or_None)      (notebook 01, unchanged except for the XGB training device switch)
    if name == "RF":
        return RandomForestClassifier(**RF_PARAMS).fit(Xtr, ytr), None
    if name == "SVM":
        idx = np.arange(len(ytr))
        if len(idx) > SVM_MAX_TRAIN_WINDOWS:
            idx, _ = train_test_split(idx, train_size=SVM_MAX_TRAIN_WINDOWS, stratify=ytr, random_state=SEED + fold)
        sc = StandardScaler().fit(Xtr[idx])
        return SVC(**SVM_PARAMS).fit(sc.transform(Xtr[idx]), ytr[idx]), sc
    if name == "XGB":
        params = dict(XGB_PARAMS)
        if XGB_DEVICE_TRAIN == "cuda": params["device"] = "cuda"
        m = xgb.XGBClassifier(objective="multi:softprob", **params)
        m.fit(Xtr, ytr, sample_weight=balanced_sample_weights(ytr, K))
        return m, None
    raise ValueError(name)

def predict(name, m, sc, X):
    Z = sc.transform(X) if sc is not None else X
    return m.predict(Z)

def complexity(name, m, n_feat):
    buf = io.BytesIO(); joblib.dump(m, buf, compress=0); size = buf.tell()
    if name == "RF":
        nodes = int(sum(e.tree_.node_count for e in m.estimators_)); return size, nodes, len(m.estimators_)
    if name == "XGB":
        dump = m.get_booster().get_dump(); return size, int(sum(d.count("\n") for d in dump)), len(dump)
    if name == "SVM":
        nsv = int(m.n_support_.sum()); return size, int(nsv * n_feat + m.dual_coef_.size + m.intercept_.size), nsv

if HAVE_TORCH:
    class CNN1D(nn.Module):
        def __init__(self, in_ch, n_classes, widths=(32, 64, 128), kernels=(5, 5, 3), dropout=0.3):
            super().__init__()
            layers, c = [], in_ch
            for i, (w, ks) in enumerate(zip(widths, kernels)):
                layers += [nn.Conv1d(c, w, ks, padding=ks // 2, bias=False), nn.BatchNorm1d(w), nn.ReLU()]
                if i < len(widths) - 1: layers.append(nn.MaxPool1d(2))
                c = w
            self.features = nn.Sequential(*layers)
            self.pool = nn.AdaptiveAvgPool1d(1)
            self.drop = nn.Dropout(dropout)
            self.fc = nn.Linear(c, n_classes)
        def forward(self, x):                      # x: (B, in_ch, T)
            return self.fc(self.drop(self.pool(self.features(x)).squeeze(-1)))

    def count_params(m): return int(sum(p.numel() for p in m.parameters()))

    def count_macs(m, in_ch, T=W):
        tot = [0]
        def hook(mod, inp, out):
            if isinstance(mod, nn.Conv1d): tot[0] += out.numel() * (mod.in_channels // mod.groups) * mod.kernel_size[0]
            elif isinstance(mod, nn.Linear): tot[0] += mod.in_features * mod.out_features
        hs = [mod.register_forward_hook(hook) for mod in m.modules() if isinstance(mod, (nn.Conv1d, nn.Linear))]
        was = m.training; m.eval()
        with torch.no_grad(): m(torch.zeros(1, in_ch, T, device=next(m.parameters()).device))
        m.train(was)
        for h in hs: h.remove()
        return int(tot[0])

    def serialized_size(m):
        buf = io.BytesIO(); torch.save(m.state_dict(), buf); return buf.tell()

    def balanced_class_weights(y, K):
        cnt = np.bincount(y, minlength=K).astype(float)
        return np.where(cnt > 0, len(y) / (K * np.maximum(cnt, 1)), 0.0)

    def train_cnn(Xtr, ytr, n_ch, seed, epochs):
        # notebook 02, unchanged (the label-shuffle option is not needed here)
        torch.manual_seed(seed); np.random.seed(seed)
        model = CNN1D(n_ch, K, **CNN_PARAMS).to(DEVICE)
        cw = torch.tensor(balanced_class_weights(ytr, K), dtype=torch.float32, device=DEVICE)
        yt = torch.from_numpy(ytr).to(DEVICE)
        bs = TRAIN_PARAMS["batch_size"]; N = len(yt); steps = max(1, N // bs)
        opt = torch.optim.AdamW(model.parameters(), lr=TRAIN_PARAMS["max_lr"], weight_decay=TRAIN_PARAMS["weight_decay"])
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=TRAIN_PARAMS["max_lr"], total_steps=epochs * steps)
        lossf = nn.CrossEntropyLoss(weight=cw)
        g = torch.Generator(device=DEVICE.type); g.manual_seed(seed)
        last = float("nan")
        for ep in range(epochs):
            model.train()
            perm = torch.randperm(N, device=DEVICE, generator=g)
            run = torch.zeros((), device=DEVICE)
            for i in range(steps):
                idx = perm[i * bs:(i + 1) * bs]
                opt.zero_grad(set_to_none=True)
                loss = lossf(model(Xtr[idx]), yt[idx])
                loss.backward(); opt.step(); sched.step()
                run += loss.detach()
            last = float(run / steps)
        return model, last

    @torch.no_grad()
    def predict_cnn(model, X, bs=4096):
        model.eval()
        return torch.cat([model(X[i:i + bs]).argmax(1) for i in range(0, len(X), bs)]).cpu().numpy()

    def cnn_prepare(D):
        # normalisation statistics from the TRAINING subjects only (as notebook 02's prepare_fold)
        Xp_t = torch.from_numpy(D["X"]).to(DEVICE)
        tr_i = torch.from_numpy(np.flatnonzero(D["tr"])).to(DEVICE); te_i = torch.from_numpy(np.flatnonzero(D["te"])).to(DEVICE)
        Xtr_raw = Xp_t[tr_i]
        mu = Xtr_raw.mean(dim=(0, 2), keepdim=True); sd = Xtr_raw.std(dim=(0, 2), keepdim=True).clamp_min(1e-6)
        out = dict(Xtr=(Xtr_raw - mu) / sd, Xte=(Xp_t[te_i] - mu) / sd, mu=mu.flatten().cpu().numpy(), sd=sd.flatten().cpu().numpy())
        del Xp_t, Xtr_raw
        return out

    _m = CNN1D(6, K, **CNN_PARAMS)
    log("CNN (6 channels):", count_params(_m), "parameters,", count_macs(_m, 6), "MACs/window,", serialized_size(_m) // 1024, "KB")
    for _c in [1, 2, 3]:
        _mm = CNN1D(_c, K, **CNN_PARAMS); log(f"   {_c} channel(s): {count_params(_mm)} parameters, {count_macs(_mm, _c)} MACs/window")
    del _m

## 5 · The machine, calibration and the timing harness

The CPU is recorded in full (`cpu_info.txt`). A small fixed benchmark (a 256 x 256 single-precision matrix product on one thread) is timed before every unit and stored in the row; if the machine slows down or speeds up during the run it shows up in the quality checks. The harness times every call twice over: preparation and inference separately, and the whole call end to end.

In [ ]:
def cpu_info():
    model, mhz = None, None
    try:
        for line in open("/proc/cpuinfo"):
            l = line.lower()
            if model is None and l.startswith("model name"): model = line.split(":", 1)[1].strip()
            if mhz is None and l.startswith("cpu mhz"):
                try: mhz = float(line.split(":", 1)[1])
                except Exception: pass
    except Exception:
        pass
    gov = None
    try: gov = open("/sys/devices/system/cpu/cpu0/cpufreq/scaling_governor").read().strip()
    except Exception: pass
    try: usable = len(os.sched_getaffinity(0))
    except Exception: usable = os.cpu_count()
    lscpu = ""
    if shutil.which("lscpu"):
        try: lscpu = subprocess.run(["lscpu"], capture_output=True, text=True, timeout=15).stdout
        except Exception: pass
    return dict(cpu_model=model or platform.processor() or "n/a", cpu_mhz=mhz, governor=gov or "n/a",
                logical_cpus=os.cpu_count(), usable_cpus=usable, lscpu=lscpu)

CPU = cpu_info()
MACHINE_ID = hashlib.sha1(f"{CPU['cpu_model']}|{CPU['logical_cpus']}|{CPU['usable_cpus']}".encode()).hexdigest()[:12]
SESSION_ID = pd.Timestamp.now("UTC").strftime("%Y%m%dT%H%M%SZ")
(RUN_DIR / "cpu_info.txt").write_text(json.dumps({k: v for k, v in CPU.items() if k != "lscpu"}, indent=2) + "\n\n" + CPU["lscpu"])
log("CPU:", CPU["cpu_model"], "| logical:", CPU["logical_cpus"], "| usable:", CPU["usable_cpus"], "| governor:", CPU["governor"],
    "| machine_id:", MACHINE_ID, "| session:", SESSION_ID)

def calibrate_ms(reps=15):
    a = np.random.RandomState(0).rand(256, 256).astype(np.float32); b = np.ascontiguousarray(a.T)
    ts = []
    with threadpool_limits(limits=1):
        for _ in range(3): a @ b
        for _ in range(reps):
            t = time.perf_counter_ns(); a @ b; ts.append(time.perf_counter_ns() - t)
    return float(np.median(ts)) / 1e6

pc_ns = time.perf_counter_ns

def run_timing(step_prep, step_infer, rows):
    # step_prep(r0) -> payload ; step_infer(payload) -> None.  Per call: preparation, inference and total (ns -> ms).
    for r0 in itertools.islice(itertools.cycle(rows), N_WARMUP):
        step_infer(step_prep(r0))
    gc.collect(); gc.disable()
    prep, infer, total, pass_med = [], [], [], []
    try:
        for _ in range(N_PASSES):
            pt = []
            for r0 in rows:
                a = pc_ns(); x = step_prep(r0); b = pc_ns(); step_infer(x); c = pc_ns()
                prep.append(b - a); infer.append(c - b); total.append(c - a); pt.append(c - a)
            pass_med.append(float(np.median(pt)))
    finally:
        gc.enable()
    return np.array(prep) / 1e6, np.array(infer) / 1e6, np.array(total) / 1e6, np.array(pass_med) / 1e6

def summarise(prep, infer, total, pass_med):
    q = lambda a, p: float(np.percentile(a, p))
    return dict(n_timings=int(len(total)),
                prep_median_ms=float(np.median(prep)), prep_iqr_ms=q(prep, 75) - q(prep, 25),
                infer_median_ms=float(np.median(infer)), infer_iqr_ms=q(infer, 75) - q(infer, 25),
                total_median_ms=float(np.median(total)), total_iqr_ms=q(total, 75) - q(total, 25),
                total_p5_ms=q(total, 5), total_p95_ms=q(total, 95), total_p99_ms=q(total, 99), total_mean_ms=float(np.mean(total)),
                pass_median_range_pct=100.0 * float((pass_med.max() - pass_med.min()) / max(np.median(total), 1e-12)))

def time_classical(mname, model, sc, ch, rows):
    chi = [IMU.index(c) for c in ch]
    if mname == "RF": model.set_params(n_jobs=1)
    if mname == "XGB":
        model.set_params(n_jobs=1)
        try: model.set_params(device="cpu")
        except Exception: pass
        assert model.get_params().get("device") in (None, "cpu"), "XGBoost inference is not on the CPU"
    if mname in ("RF", "XGB"):
        assert model.get_params()["n_jobs"] == 1, "inference is not single-threaded"
    def prep(r0):
        return window_features(sig[r0:r0 + W][None, :, chi], ch)[0]       # slice -> 15 features per channel
    def infer(fx):
        predict(mname, model, sc, fx)                                       # (SVM: scaler) -> predict
    with threadpool_limits(limits=1):
        return run_timing(prep, infer, rows)

def time_cnn(model, mu, sd, ch_idx, rows):
    m = copy.deepcopy(model).to("cpu").eval()
    muc, sdc = mu[ch_idx][:, None], sd[ch_idx][:, None]
    prev = torch.get_num_threads(); torch.set_num_threads(1)
    def prep(r0):
        return torch.from_numpy(((sig[r0:r0 + W][:, ch_idx].T - muc) / sdc).astype(np.float32))[None]   # slice -> z-score
    def infer(x):
        int(m(x).argmax(1))                                                                                # forward -> argmax
    try:
        with torch.inference_mode(), threadpool_limits(limits=1):
            return run_timing(prep, infer, rows)
    finally:
        torch.set_num_threads(prev)

log("Calibration micro-benchmark (256x256 matmul, 1 thread):", round(calibrate_ms(), 4), "ms")

## 6 · Run integrity (config hash, machine guard) and resume

Results are appended unit by unit. A configuration hash refuses to mix results made with different settings; the machine guard refuses to mix machines.

In [ ]:
design = {"window_samples": W, "step": S, "fs": FS, "labels": labels, "subjects": subjects, "lat_fold": LAT_FOLD,
          "rf": RF_PARAMS, "svm": SVM_PARAMS, "svm_cap": SVM_MAX_TRAIN_WINDOWS, "xgb": XGB_PARAMS, "cnn": CNN_PARAMS, "train": TRAIN_PARAMS,
          "n_time_windows": N_TIME_WINDOWS, "n_passes": N_PASSES, "n_warmup": N_WARMUP, "data_sha256": DATA_SHA256,
          "windows_sha256": WINDOWS_SHA256, "smoke": SMOKE_TEST, "seed": SEED}
cfg_hash = hashlib.sha256(json.dumps(design, sort_keys=True, default=str).encode()).hexdigest()
cfg_path = RUN_DIR / "run_config.json"
if cfg_path.exists():
    old = json.load(open(cfg_path))
    if old.get("hash") != cfg_hash and not FORCE_NEW_RUN:
        raise RuntimeError("Existing results were produced with DIFFERENT settings. Use a new RESULTS_DIR, or set FORCE_NEW_RUN=1 "
                           "after deleting results.csv yourself.")
json.dump({"hash": cfg_hash, "design": design}, open(cfg_path, "w"), indent=2, default=str)

RES_PATH = RUN_DIR / "results.csv"
done = set()
if RES_PATH.exists():
    _d = pd.read_csv(RES_PATH, usecols=["position", "config", "model", "machine_id"])
    done = set(map(tuple, _d[["position", "config", "model"]].to_numpy().tolist()))
    _other = sorted(set(_d.machine_id) - {MACHINE_ID})
    if _other and not ALLOW_MIXED_MACHINES:
        raise RuntimeError(f"results.csv already holds units measured on another machine ({_other}); this machine is {MACHINE_ID}. "
                           "Latencies from different machines must not be mixed. Start a new RESULTS_DIR, or set ALLOW_MIXED_MACHINES=1 on purpose.")
    log("Units already completed:", len(done), "| machines in file:", sorted(set(_d.machine_id)))
else:
    log("Units already completed: 0")

## 7 · Main loop (resumable, one row per unit)

For each position the held-out subject's windows are fixed, the 300 windows to time are chosen evenly across them, and for every (model, configuration) one model is trained on the training subjects of the fold, evaluated once for a sanity macro-F1, and then timed on the CPU. Finished units are skipped on re-run. A fixed column list keeps all four models in one consistent `results.csv`.

In [ ]:
COLUMNS = ["position", "config", "group", "model", "n_channels", "channels", "data_rate_hz", "fold", "test_subject",
           "n_train_windows", "n_test_windows", "n_timed_windows", "n_passes", "n_warmup", "n_timings", "n_features",
           "prep_median_ms", "prep_iqr_ms", "infer_median_ms", "infer_iqr_ms", "total_median_ms", "total_iqr_ms",
           "total_p5_ms", "total_p95_ms", "total_p99_ms", "total_mean_ms", "pass_median_range_pct",
           "model_size_bytes", "n_params", "n_params_proxy", "n_trees_or_sv", "macs_per_window",
           "train_time_s", "final_train_loss", "macro_f1_fold", "train_device", "cnn_weights",
           "calib_ms", "machine_id", "cpu_model", "session_id", "timestamp_utc"]
if RES_PATH.exists():
    assert list(pd.read_csv(RES_PATH, nrows=0).columns) == COLUMNS, "results.csv has a different column layout; use a new RESULTS_DIR"

TOTAL_UNITS = len(POS_LIST) * len(CONFIGS) * len(MODELS)
SEL_KEYS = {(p, c, m) for p in POS_LIST for c in CONFIGS for m in MODELS}
n_new, t_start, STOPPED_EARLY = 0, time.time(), False
log(f"Units in this selection: {TOTAL_UNITS} | already done: {len(done & SEL_KEYS)}")

def append_row(row):
    pd.DataFrame([{c: row.get(c, np.nan) for c in COLUMNS}]).to_csv(RES_PATH, mode="a", header=not RES_PATH.exists(), index=False)

for p in POS_LIST:
    if STOPPED_EARLY: break
    if all((p, c, m) in done for c in CONFIGS for m in MODELS): continue
    D = POS_DATA[p]
    te = D["subj"] == FD["test_subject"]; tr = ~te
    assert FD["test_subject"] not in set(D["subj"][tr]), "test subject present in training windows"
    assert set(D["subj"][tr]) == set(FD["train_subjects"]), "training subjects differ from the fold definition"
    ytr, yte = D["y"][tr], D["y"][te]
    assert len(np.unique(ytr)) == K, "a class is missing from the training subjects"
    present = np.unique(yte)
    te_rows_all = D["starts"][te]
    te_rows = te_rows_all[np.unique(np.linspace(0, len(te_rows_all) - 1, N_TIME_WINDOWS).round().astype(int))]
    if not SMOKE_TEST:
        assert len(te_rows) >= 200, "fewer than 200 distinct test windows to time"
    Ftr_all, Fte_all = D["F"][tr], D["F"][te]
    D["tr"], D["te"] = tr, te
    CNNP = None
    log(f"--- position {p}: {int(tr.sum())} training windows, {int(te.sum())} held-out windows, {len(te_rows)} timed windows")

    for mname in MODELS:
        if STOPPED_EARLY: break
        for cname, ch in CONFIGS.items():
            key = (p, cname, mname)
            if key in done: continue
            if TIME_BUDGET_S is not None and time.time() - T0 > TIME_BUDGET_S:
                STOPPED_EARLY = True
                log(f"TIME_BUDGET_MIN reached after {(time.time() - T0) / 60:.1f} min: stopping cleanly. Re-run on the SAME machine to resume.")
                break
            ch_idx = [IMU.index(c) for c in ch]
            calib = calibrate_ms()
            row = dict(position=p, config=cname, group=GROUP_OF[cname], model=mname, n_channels=len(ch), channels="".join(ch),
                       data_rate_hz=len(ch) * FS, fold=LAT_FOLD, test_subject=FD["test_subject"], n_train_windows=int(tr.sum()),
                       n_test_windows=int(te.sum()), n_timed_windows=int(len(te_rows)), n_passes=N_PASSES, n_warmup=N_WARMUP)
            if mname == "CNN":
                if CNNP is None:
                    CNNP = cnn_prepare(D)
                    _tr0 = D["X"][D["tr"]].astype(np.float64)
                    assert np.allclose(CNNP["mu"], _tr0.mean(axis=(0, 2)), atol=1e-4), "normalisation mean is not the training-subject mean"
                    assert np.allclose(CNNP["sd"], _tr0.std(axis=(0, 2), ddof=1), atol=1e-3), "normalisation std is not the training-subject std"
                    del _tr0
                ch_t = torch.tensor(ch_idx, device=DEVICE)
                Xtr_c = CNNP["Xtr"].index_select(1, ch_t).contiguous(); Xte_c = CNNP["Xte"].index_select(1, ch_t).contiguous()
                assert Xtr_c.shape[1] == len(ch) and Xte_c.shape[1] == len(ch)
                t0 = time.time()
                if CNN_DO_TRAIN:
                    model, last_loss = train_cnn(Xtr_c, ytr, len(ch), SEED + LAT_FOLD, TRAIN_PARAMS["epochs"])
                    if DEVICE.type == "cuda": torch.cuda.synchronize()
                else:
                    torch.manual_seed(SEED + LAT_FOLD)
                    model = CNN1D(len(ch), K, **CNN_PARAMS).to(DEVICE); last_loss = float("nan")
                t_fit = time.time() - t0
                f1 = float(f1_score(yte, predict_cnn(model, Xte_c), labels=present, average="macro", zero_division=0)) if CNN_DO_TRAIN else float("nan")
                row.update(n_features=len(ch) * W, macro_f1_fold=f1, train_time_s=t_fit, final_train_loss=last_loss,
                           model_size_bytes=serialized_size(model), n_params=count_params(model), macs_per_window=count_macs(model, len(ch)),
                           train_device=DEVICE.type if CNN_DO_TRAIN else "none", cnn_weights=CNN_WEIGHTS)
                timing = time_cnn(model, CNNP["mu"], CNNP["sd"], ch_idx, te_rows)
                del Xtr_c, Xte_c
            else:
                cols = [i for i, n in enumerate(D["names"]) if n.split("__")[0] in ch]
                assert len(cols) == len(FEATS) * len(ch)
                Xtr, Xte = Ftr_all[:, cols], Fte_all[:, cols]
                t0 = time.time(); model, sc = fit_model(mname, Xtr, ytr, LAT_FOLD); t_fit = time.time() - t0
                f1 = float(f1_score(yte, predict(mname, model, sc, Xte), labels=present, average="macro", zero_division=0))
                size, n_par, n_units = complexity(mname, model, len(cols))
                row.update(n_features=len(cols), macro_f1_fold=f1, train_time_s=t_fit, model_size_bytes=size, n_params_proxy=n_par,
                           n_trees_or_sv=n_units, train_device=XGB_DEVICE_TRAIN if mname == "XGB" else "cpu", cnn_weights="n/a")
                time.sleep(0.2)                        # let any training thread pools settle before timing
                timing = time_classical(mname, model, sc, ch, te_rows)
            row.update(summarise(*timing))
            row.update(calib_ms=calib, machine_id=MACHINE_ID, cpu_model=CPU["cpu_model"], session_id=SESSION_ID,
                       timestamp_utc=pd.Timestamp.now("UTC").isoformat())
            append_row(row)
            done.add(key); n_new += 1
            del model
            el = time.time() - t_start; remaining = TOTAL_UNITS - len(done & SEL_KEYS)
            log(f"[{len(done & SEL_KEYS)}/{TOTAL_UNITS}] {p} {mname} {cname}: total={row['total_median_ms']:.3f} ms "
                f"(prep {row['prep_median_ms']:.3f} + infer {row['infer_median_ms']:.3f}), pass-range {row['pass_median_range_pct']:.1f}%, "
                f"fit={t_fit:.1f}s | elapsed {el / 60:.1f} min, ETA ~ {el / n_new * remaining / 60:.1f} min")
    del CNNP
    if DEVICE is not None and DEVICE.type == "cuda":
        torch.cuda.empty_cache()
log("Loop finished. New units this session:", n_new, "| stopped early:", STOPPED_EARLY)

## 8 · Completeness check

Analysis runs on the units of this selection. If the run is incomplete (time budget, interruption) the tables and figures are skipped unless `ANALYZE_PARTIAL=1`; re-run on the same machine to finish.

In [ ]:
res_all = pd.read_csv(RES_PATH)
assert list(res_all.columns) == COLUMNS
assert res_all.duplicated(subset=["position", "config", "model"]).sum() == 0, "duplicated units in results.csv"
res = res_all[[(a, b, c) in SEL_KEYS for a, b, c in zip(res_all.position, res_all.config, res_all.model)]].reset_index(drop=True)
COMPLETE = len(res) == TOTAL_UNITS
log("Rows for this selection:", len(res), "of", TOTAL_UNITS, "| complete:", COMPLETE)
if REQUIRE_COMPLETE and not COMPLETE:
    raise RuntimeError("REQUIRE_COMPLETE=1 and units are missing: re-run this notebook (same machine) to finish.")
if not COMPLETE:
    log("INCOMPLETE: re-run to finish the remaining units.")
DO_ANALYSIS = (COMPLETE or ANALYZE_PARTIAL) and len(res) > 0
if not DO_ANALYSIS:
    log("Tables and figures are skipped for an incomplete run (set ANALYZE_PARTIAL=1 to force them).")
else:
    log("Machines in the analysed rows:", sorted(res.machine_id.unique()), "| sessions:", sorted(res.session_id.unique()))
    if res.machine_id.nunique() > 1:
        log("WARNING: more than one machine in the analysed rows (ALLOW_MIXED_MACHINES was used); latencies are NOT mutually comparable.")

## 9 · Quality checks and the join with the accuracy grid

Three checks on the measurement, none of which changes a reported number:

- **Stability:** the spread of the three per-pass medians, relative to the median (a noisy neighbour on a shared cloud CPU shows up here).
- **Drift:** the calibration micro-benchmark over the run and per session.
- **Representativeness:** model size here against the median over the 10 folds of the accuracy grid, and the fold macro-F1 of the model timed here against the same fold in the grid (equal up to library and GPU nondeterminism; large gaps would mean the timed model is not the modelled one).

The same join provides the 10-fold mean macro-F1 (seed 42, immutable grid) used by the Pareto view.

In [ ]:
GRID = pd.DataFrame(); GRID_SUM = pd.DataFrame(); QC = pd.DataFrame(); QC_SUMMARY = {}
if DO_ANALYSIS:
    frames = []
    for gname, gdir in GRID_DIRS.items():
        if gdir is None: continue
        hdr = pd.read_csv(Path(gdir) / "results.csv", nrows=0).columns
        use = [c for c in ["position", "config", "model", "fold", "seed", "macro_f1", "model_size_bytes"] if c in hdr]
        g = pd.read_csv(Path(gdir) / "results.csv", usecols=use)
        if "seed" in g.columns: g = g[g["seed"] == SEED]
        g["source"] = gname; frames.append(g)
    if frames:
        GRID = pd.concat(frames, ignore_index=True)
        _dup = GRID.duplicated(subset=["position", "model", "config", "fold"]).sum()
        assert _dup == 0, f"{_dup} duplicated (position, model, config, fold) rows across the accuracy grids (seed {SEED})"
        GRID_SUM = (GRID.groupby(["position", "model", "config"])
                    .agg(n_folds=("fold", "nunique"), macro_f1_mean=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
                         grid_size_median=("model_size_bytes", "median")).reset_index())
        _f = GRID[GRID["fold"] == LAT_FOLD][["position", "model", "config", "macro_f1"]].rename(columns={"macro_f1": "macro_f1_grid_fold"})
        GRID_SUM = GRID_SUM.merge(_f, on=["position", "model", "config"], how="left")
        log("Accuracy grid joined:", len(GRID_SUM), "(position, model, config) cells from", sorted(GRID.source.unique()))
        if (GRID_SUM.n_folds != len(subjects)).any():
            log("WARNING: some grid cells do not have all", len(subjects), "folds; their mean macro-F1 is over fewer folds.")
    else:
        log("No accuracy grid found: size/F1 representativeness checks and the Pareto view are skipped.")

    QC = res[["position", "model", "config", "group", "n_channels", "pass_median_range_pct", "calib_ms", "macro_f1_fold",
              "model_size_bytes", "session_id", "machine_id", "cnn_weights"]].copy()
    if len(GRID_SUM):
        QC = QC.merge(GRID_SUM[["position", "model", "config", "grid_size_median", "macro_f1_grid_fold"]], on=["position", "model", "config"], how="left")
        QC["size_ratio_vs_grid_median"] = QC["model_size_bytes"] / QC["grid_size_median"]
        QC["f1_abs_diff_vs_grid_fold"] = (QC["macro_f1_fold"] - QC["macro_f1_grid_fold"]).abs()
    QC_SUMMARY = dict(
        n_units=int(len(QC)),
        pass_range_pct_median=float(QC.pass_median_range_pct.median()), pass_range_pct_max=float(QC.pass_median_range_pct.max()),
        share_units_pass_range_over_10pct=float((QC.pass_median_range_pct > 10).mean()),
        calib_ms_median=float(QC.calib_ms.median()), calib_ms_cv=float(QC.calib_ms.std() / QC.calib_ms.mean()) if len(QC) > 1 else 0.0,
        calib_ms_median_by_session={k: float(v) for k, v in QC.groupby("session_id").calib_ms.median().items()})
    if QC_SUMMARY["calib_ms_cv"] > 0.15:
        log("WARNING: the calibration benchmark varied by more than 15 % (CV) during the run: the machine was not steady. Interpret small differences with care.")
    if QC_SUMMARY["share_units_pass_range_over_10pct"] > 0.10:
        log("WARNING: more than 10 % of units show a spread above 10 % between pass medians (noisy machine).")
    if "size_ratio_vs_grid_median" in QC.columns:
        _sr = QC.groupby("model").size_ratio_vs_grid_median.agg(["median", "min", "max"]).round(3)
        QC_SUMMARY["size_ratio_vs_grid_median_by_model"] = json.loads(_sr.to_json(orient="index"))
        display(_sr)
        _fd = QC.dropna(subset=["f1_abs_diff_vs_grid_fold"]).groupby("model").f1_abs_diff_vs_grid_fold.agg(["median", "max"]).round(4)
        QC_SUMMARY["fold_macro_f1_abs_diff_vs_grid_by_model"] = json.loads(_fd.to_json(orient="index"))
        display(_fd)
        if len(_fd) and (_fd["max"] > 0.05).any():
            log("WARNING: fold macro-F1 of a timed model differs from the grid by more than 0.05 for", list(_fd.index[_fd["max"] > 0.05]),
                "(expected only for reduced smoke-test models or different library versions).")
    QC.to_csv(TABLE_DIR / "TableL6_measurement_quality.csv", index=False)
    json.dump(QC_SUMMARY, open(CHECK_DIR / "latency_qc.json", "w"), indent=2)
    log("Quality summary:", json.dumps(QC_SUMMARY)[:900])

## 10 · Tables

`TableL1` per unit; `TableL2` pooled by channel count (median, quartiles, min to max over all units of that size, pure and mixed); `TableL3` the same by position; `TableL4` the analytic data-volume model (D2); `TableL5` the Pareto flags (D4). Duty cycle is latency divided by the 2 s hop, the share of one core of **this** CPU. It is not energy.

In [ ]:
TABLES = {}
if DO_ANALYSIS:
    lat = res.copy()
    lat["complexity_count"] = np.where(lat.model == "CNN", lat.n_params, lat.n_params_proxy)
    lat["complexity_kind"] = lat.model.map({"CNN": "parameters", "RF": "tree nodes", "XGB": "tree nodes", "SVM": "SVM parameters (SV x features + dual coefficients)"})
    lat["size_kb"] = lat.model_size_bytes / 1024.0
    lat["duty_cycle_pct"] = 100.0 * lat.total_median_ms / HOP_MS
    lat["prep_share_pct"] = 100.0 * lat.prep_median_ms / (lat.prep_median_ms + lat.infer_median_ms)
    if "6-axis" in set(lat.config):
        _b = lat[lat.config == "6-axis"].set_index(["position", "model"])
        for col in ["total_median_ms", "prep_median_ms", "infer_median_ms", "size_kb"]:
            ref_v = lat.set_index(["position", "model"]).index.map(_b[col].to_dict())
            lat[f"{col}_vs_6axis_pct"] = 100.0 * (lat[col].to_numpy() / np.asarray(ref_v, dtype=float) - 1.0)
    L1_cols = ["position", "model", "config", "group", "n_channels", "channels", "data_rate_hz", "n_features", "total_median_ms", "total_iqr_ms",
               "total_p5_ms", "total_p95_ms", "total_p99_ms", "total_mean_ms", "prep_median_ms", "infer_median_ms", "prep_share_pct",
               "duty_cycle_pct", "size_kb", "complexity_count", "complexity_kind", "macs_per_window", "n_trees_or_sv", "train_time_s",
               "total_median_ms_vs_6axis_pct", "prep_median_ms_vs_6axis_pct", "infer_median_ms_vs_6axis_pct", "size_kb_vs_6axis_pct",
               "pass_median_range_pct", "n_timings", "train_device", "cnn_weights", "cpu_model", "machine_id"]
    L1 = lat[[c for c in L1_cols if c in lat.columns]].sort_values(["position", "model", "n_channels", "config"], ascending=[True, True, False, True])
    L1.to_csv(TABLE_DIR / "TableL1_latency_by_unit.csv", index=False); TABLES["L1"] = L1

    METRICS = [("total_median_ms", "total_ms"), ("prep_median_ms", "prep_ms"), ("infer_median_ms", "infer_ms"),
               ("size_kb", "size_kb"), ("complexity_count", "complexity_count"), ("duty_cycle_pct", "duty_cycle_pct")]
    def pooled(df, keys):
        rows = []
        for k, g in df.groupby(keys):
            k = k if isinstance(k, tuple) else (k,)
            r = dict(zip(keys, k)); r["n_units"] = int(len(g)); r["n_configs"] = int(g.config.nunique())
            for col, nm in METRICS:
                v = g[col].dropna()
                r[f"{nm}_median"] = float(v.median()) if len(v) else np.nan
                r[f"{nm}_q25"] = float(v.quantile(0.25)) if len(v) else np.nan
                r[f"{nm}_q75"] = float(v.quantile(0.75)) if len(v) else np.nan
                r[f"{nm}_min"] = float(v.min()) if len(v) else np.nan
                r[f"{nm}_max"] = float(v.max()) if len(v) else np.nan
            r["macs_median"] = float(g.macs_per_window.dropna().median()) if g.macs_per_window.notna().any() else np.nan
            rows.append(r)
        out = pd.DataFrame(rows)
        grp = [c for c in keys if c != "n_channels"]
        base = out[out.n_channels == 6]
        for _, nm in METRICS:
            col = f"{nm}_median"
            if len(base):
                bmap = base.set_index(grp)[col] if grp else None
                if grp:
                    refs = out.set_index(grp).index.map(bmap.to_dict())
                else:
                    refs = [float(base[col].iloc[0])] * len(out)
                out[f"{nm}_median_vs_6ch"] = out[col].to_numpy() / np.asarray(refs, dtype=float)
        return out.sort_values(grp + ["n_channels"], ascending=[True] * len(grp) + [False])
    L2 = pooled(lat, ["model", "n_channels"]); L2.to_csv(TABLE_DIR / "TableL2_cost_by_channel_count.csv", index=False); TABLES["L2"] = L2
    L3 = pooled(lat, ["position", "model", "n_channels"]); L3.to_csv(TABLE_DIR / "TableL3_cost_by_position_channel_count.csv", index=False); TABLES["L3"] = L3
    show = L2[["model", "n_channels", "n_units", "total_ms_median", "total_ms_min", "total_ms_max", "prep_ms_median", "infer_ms_median", "size_kb_median", "complexity_count_median"]]
    display(show.round(3))

    # ---- D2: analytic data-volume model (no measurement)
    dv = []
    for cname, ch in CONFIGS.items():
        n = len(ch); sps = n * FS
        r = dict(config=cname, group=GROUP_OF[cname], n_channels=n, channels="".join(ch), samples_per_s=sps, window_samples=W, hop_s=HOP_S,
                 window_values=W * n, reduction_vs_6axis_pct=100.0 * (1 - n / 6.0))
        for bits in (16, 32):
            B = bits // 8
            r[f"bytes_per_s_{bits}bit"] = sps * B
            r[f"window_payload_bytes_{bits}bit"] = W * n * B
            r[f"MB_per_hour_{bits}bit"] = sps * B * 3600 / 1e6
            r[f"MB_per_day_{bits}bit"] = sps * B * 86400 / 1e6
        dv.append(r)
    L4 = pd.DataFrame(dv).sort_values(["n_channels", "config"], ascending=[False, True])
    L4.to_csv(TABLE_DIR / "TableL4_data_volume.csv", index=False); TABLES["L4"] = L4

    # ---- D4: Pareto flags (macro-F1 from the immutable accuracy grid; x lower is better, F1 higher is better)
    def pareto_flags(df, xcol, ycol):
        x = df[xcol].to_numpy(float); y = df[ycol].to_numpy(float); flag = np.zeros(len(df), dtype=bool)
        for i in range(len(df)):
            if np.isnan(x[i]) or np.isnan(y[i]): continue
            flag[i] = not ((x <= x[i]) & (y >= y[i]) & ((x < x[i]) | (y > y[i]))).any()
        return flag
    L5 = pd.DataFrame()
    if len(GRID_SUM):
        L5 = lat[["position", "model", "config", "group", "n_channels", "data_rate_hz", "total_median_ms", "size_kb"]].merge(
            GRID_SUM[["position", "model", "config", "n_folds", "macro_f1_mean"]], on=["position", "model", "config"], how="left")
        L5["pareto_vs_data_rate"] = False; L5["pareto_vs_latency"] = False
        for (pp, mm), idx in L5.groupby(["position", "model"]).groups.items():
            sub = L5.loc[idx]
            L5.loc[idx, "pareto_vs_data_rate"] = pareto_flags(sub, "data_rate_hz", "macro_f1_mean")
            L5.loc[idx, "pareto_vs_latency"] = pareto_flags(sub, "total_median_ms", "macro_f1_mean")
        L5["pareto_vs_data_rate"] = L5["pareto_vs_data_rate"].astype(bool); L5["pareto_vs_latency"] = L5["pareto_vs_latency"].astype(bool)
        L5.to_csv(TABLE_DIR / "TableL5_pareto.csv", index=False); TABLES["L5"] = L5
        log("Pareto view built for", int(L5.macro_f1_mean.notna().sum()), "units with accuracy results.")
    else:
        log("TableL5 (Pareto) skipped: no accuracy grid found.")

    _note = (f"All latencies in this folder: single window, batch size 1, 1 CPU thread, median of {N_TIME_WINDOWS * N_PASSES} timed calls per unit "
             f"(IQR, 5th, 95th and 99th percentile also reported), preparation (slice, features or z-score) plus inference, measured in one session on "
             f"{CPU['cpu_model']} ({CPU['usable_cpus']} usable logical CPUs). One model per unit, trained on the training subjects of fold {LAT_FOLD}. "
             f"The numbers include the call overhead of scikit-learn, XGBoost and PyTorch. They are NOT edge-device latencies and NOT energy. "
             f"Duty cycle = latency / {HOP_MS:.0f} ms hop = share of one core of this CPU. Data volume is an analytic model of raw sample values only "
             f"(16-bit and 32-bit), without headers, timestamps, transport overhead or compression.")
    (CHECK_DIR / "latency_note.txt").write_text(_note)
    log("Tables written to", TABLE_DIR)

## 11 · Figures

`FL1` latency and model size against channel count; `FL2` where the time goes (preparation versus inference); `FL3` data volume against channel count; `FL4` macro-F1 against data rate and `FL5` macro-F1 against latency, one panel per position and model, Pareto-optimal points filled. PNG (300 dpi) and PDF.

In [ ]:
FIGS = []
if DO_ANALYSIS:
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 9, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
    MODEL_COL = {"RF": "#1b9e77", "SVM": "#7570b3", "XGB": "#d95f02", "CNN": "#1f78b4"}
    GROUP_STYLE = {"P": ("o", "#1f78b4", "primary"), "A1": ("s", "#e66101", "drop-one"), "A2": ("^", "#4daf4a", "mixed pair")}
    MODELS_PRESENT = [m for m in ["RF", "SVM", "XGB", "CNN"] if m in set(res.model)]
    def savefig(fig, name):
        for ext in ("png", "pdf"):
            fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=300, bbox_inches="tight")
        FIGS.append(name); plt.show(); plt.close(fig)

    # FL1: latency and size by channel count
    L2 = TABLES["L2"]
    fig, ax = plt.subplots(1, 2, figsize=(8.4, 3.4))
    for m in MODELS_PRESENT:
        g = L2[L2.model == m].sort_values("n_channels")
        ax[0].plot(g.n_channels, g.total_ms_median, "-o", color=MODEL_COL[m], label=m)
        ax[0].fill_between(g.n_channels, g.total_ms_min, g.total_ms_max, color=MODEL_COL[m], alpha=0.12, linewidth=0)
        ax[1].plot(g.n_channels, g.size_kb_median, "-o", color=MODEL_COL[m], label=m)
        ax[1].fill_between(g.n_channels, g.size_kb_min, g.size_kb_max, color=MODEL_COL[m], alpha=0.12, linewidth=0)
    ax[0].set_yscale("log"); ax[1].set_yscale("log")
    ax[0].set_ylabel("single-window latency (ms, log)"); ax[1].set_ylabel("serialized model size (KB, log)")
    for a in ax:
        a.set_xlabel("channels"); a.set_xticks(sorted(L2.n_channels.unique()))
    ax[0].legend(frameon=False, ncol=2)
    fig.suptitle("Median over units of each channel count (band: min to max over positions and configurations)", fontsize=8)
    fig.tight_layout(); savefig(fig, "FL1_latency_and_size_vs_channels")

    # FL2: where the time goes
    fig, axes = plt.subplots(1, len(MODELS_PRESENT), figsize=(2.6 * len(MODELS_PRESENT) + 0.6, 3.2), sharey=False, squeeze=False)
    for a, m in zip(axes[0], MODELS_PRESENT):
        g = L2[L2.model == m].sort_values("n_channels")
        a.bar(g.n_channels, g.prep_ms_median, color="#bdbdbd", label="preparation")
        a.bar(g.n_channels, g.infer_ms_median, bottom=g.prep_ms_median, color=MODEL_COL[m], label="inference")
        a.set_title(m); a.set_xlabel("channels"); a.set_xticks(sorted(L2.n_channels.unique()))
    axes[0][0].set_ylabel("ms (median of the component medians)")
    axes[0][0].legend(frameon=False, fontsize=7)
    fig.tight_layout(); savefig(fig, "FL2_preparation_vs_inference")

    # FL3: data volume (analytic)
    L4 = TABLES["L4"]; by_n = L4.groupby("n_channels").first().reset_index()
    fig, ax = plt.subplots(figsize=(4.4, 3.2))
    ax.plot(by_n.n_channels, by_n.bytes_per_s_16bit, "-o", color="#1f78b4", label="16-bit samples")
    ax.plot(by_n.n_channels, by_n.bytes_per_s_32bit, "-s", color="#e66101", label="32-bit samples")
    for r in by_n.itertuples():
        ax.annotate(f"-{r.reduction_vs_6axis_pct:.0f}%" if r.reduction_vs_6axis_pct > 0 else "6-axis", (r.n_channels, r.bytes_per_s_32bit), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=7)
    ax.set_xlabel("channels"); ax.set_ylabel("raw sample data (bytes per second)"); ax.set_xticks(sorted(by_n.n_channels.unique())); ax.legend(frameon=False)
    ax.set_title("Data volume model at 20 Hz per channel", fontsize=9)
    fig.tight_layout(); savefig(fig, "FL3_data_volume_vs_channels")

    # FL4 / FL5: Pareto grids
    if "L5" in TABLES and TABLES["L5"].macro_f1_mean.notna().any():
        L5 = TABLES["L5"]; poss = [p for p in ["wrist", "pocket", "leg"] if p in set(L5.position)]
        def pareto_grid(xcol, flagcol, xlabel, logx, name):
            fig, axes = plt.subplots(len(poss), len(MODELS_PRESENT), figsize=(2.7 * len(MODELS_PRESENT) + 0.5, 2.5 * len(poss) + 0.4), squeeze=False)
            for i, pp in enumerate(poss):
                for j, mm in enumerate(MODELS_PRESENT):
                    a = axes[i][j]; g = L5[(L5.position == pp) & (L5.model == mm)].dropna(subset=["macro_f1_mean"])
                    for grp, (mk, col, lab) in GROUP_STYLE.items():
                        s = g[g.group == grp]
                        if not len(s): continue
                        xs = s[xcol].to_numpy(float)
                        if xcol == "data_rate_hz": xs = xs + {"P": -1.5, "A1": 0.0, "A2": 1.5}[grp]
                        fl = s[flagcol].to_numpy(bool)
                        a.scatter(xs[~fl], s.macro_f1_mean.to_numpy()[~fl], marker=mk, facecolors="none", edgecolors=col, s=22, linewidths=0.9)
                        a.scatter(xs[fl], s.macro_f1_mean.to_numpy()[fl], marker=mk, color=col, s=30, label=lab if (i == 0 and j == 0) else None)
                    fr = g[g[flagcol]].sort_values(xcol)
                    if len(fr): a.step(fr[xcol], fr.macro_f1_mean, where="post", color="k", linewidth=0.8, alpha=0.6)
                    if logx: a.set_xscale("log")
                    if i == 0: a.set_title(mm)
                    if j == 0: a.set_ylabel(f"{pp}\nmacro-F1")
                    if i == len(poss) - 1: a.set_xlabel(xlabel)
            axes[0][0].legend(frameon=False, fontsize=6, loc="lower right")
            fig.tight_layout(); savefig(fig, name)
        pareto_grid("data_rate_hz", "pareto_vs_data_rate", "data rate (Hz = channels x 20)", False, "FL4_pareto_f1_vs_data_rate")
        pareto_grid("total_median_ms", "pareto_vs_latency", "single-window latency (ms, log)", True, "FL5_pareto_f1_vs_latency")
    else:
        log("FL4/FL5 skipped: no accuracy grid to take macro-F1 from.")
    log("Figures written:", FIGS)

## 12 · Environment record, manifest and zip

In [ ]:
import importlib
def ver(m):
    try: return importlib.import_module(m).__version__
    except Exception: return "n/a"
env = {"python": sys.version, "platform": platform.platform(), "cpu": {k: v for k, v in CPU.items() if k != "lscpu"},
       "machine_id": MACHINE_ID, "session_id": SESSION_ID, "gpu_visible": HAS_GPU,
       "torch_device": str(DEVICE), "gpu": (torch.cuda.get_device_name(0) if (HAVE_TORCH and torch.cuda.is_available()) else None),
       "xgb_train_device": XGB_DEVICE_TRAIN, "cnn_weights": CNN_WEIGHTS, "timing_threads": 1,
       "timing_note": "single window, batch 1, 1 CPU thread; preparation + inference; shared cloud CPU; NOT edge-device latency; no energy measured",
       "lat_fold": LAT_FOLD, "n_time_windows": N_TIME_WINDOWS, "n_passes": N_PASSES, "n_warmup": N_WARMUP, "smoke_test": SMOKE_TEST,
       "models": MODELS, "positions": POS_LIST, "config_groups": CONFIG_GROUPS, "n_configs": len(CONFIGS),
       "packages": {m: ver(m) for m in ["numpy", "pandas", "sklearn", "xgboost", "torch", "threadpoolctl", "joblib", "scipy", "matplotlib", "pyarrow"]},
       "data_sha256": DATA_SHA256, "windows_sha256": WINDOWS_SHA256, "config_hash": cfg_hash, "stopped_early": STOPPED_EARLY,
       "accuracy_grids_used": {k: (str(v) if v is not None else None) for k, v in GRID_DIRS.items()},
       "timestamp_utc": pd.Timestamp.now("UTC").isoformat(), "notebook_minutes": round((time.time() - T0) / 60, 1)}
json.dump(env, open(RUN_DIR / "environment.json", "w"), indent=2, default=str)
(RUN_DIR / "run_log.txt").write_text("\n".join(LOG))

def sha_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
manifest = {str(f.relative_to(RUN_DIR)): {"bytes": f.stat().st_size, "sha256": sha_file(f)}
            for f in sorted(RUN_DIR.rglob("*")) if f.is_file() and f.name != "manifest.json"}
json.dump(manifest, open(CHECK_DIR / "manifest.json", "w"), indent=2)

zip_path = BASE / f"recgym_05_latency_{TAG}.zip"
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in RUN_DIR.rglob("*"):
        if f.is_file(): z.write(f, f"latency_{TAG}/{f.relative_to(RUN_DIR)}")
log("ZIP:", zip_path, "| MB:", round(os.path.getsize(zip_path) / 1e6, 2))
if SMOKE_TEST:
    log("Smoke run finished. Check the ETA, the pass-range and calibration lines, and that the Pareto view found the accuracy grid. "
        "Then restart and run with RUN_OVERRIDES = {\"SMOKE_TEST\": \"0\"}.")
elif not COMPLETE:
    log("Run is INCOMPLETE: re-run this notebook on the same machine to finish, then upload the zip.")
else:
    log("Complete. Upload the zip for the extended analysis (notebook 06) and the systems paragraph of the manuscript.")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    log("Not on Colab: find the zip in /kaggle/working (Output tab) or your working directory.")
(RUN_DIR / "run_log.txt").write_text("\n".join(LOG))